# OCCLUDE-only — Google Colab

Use this when the video is **already in Drive** and you only want the blur pass. It skips ELUATE entirely:

1. Install `occlude[gpu]` and repair the `onnxruntime-gpu` shadowing so InsightFace runs on CUDA.
2. Verify every stage is on CUDA (hard-stops if not).
3. Read from `MyDrive/occlude/inputs/`, write the blurred result to `MyDrive/occlude/outputs/`.

**Runtime (Pro+)**: Runtime → Change runtime type → **A100** (fall back to **L4**). Enable **background execution** so the job survives a closed tab.

**First-batch warm-up**: `torch.compile` fuses the SegFormer kernels on the first perception batch — the first ~10–30 frames crawl, then accelerate. Expected, not a hang; negligible across a feature-length file.

This notebook does **not** assert an exact file size (that check in the full-run notebook is what produced the earlier "file error"). It only checks the file exists, and cell 6 lists the inputs folder so you can confirm the exact name.

In [ ]:
# 1. Verify GPU (expect A100 / L4 on Pro+)
!nvidia-smi

In [ ]:
# 2. System deps: ffmpeg (decode + remux), libgl1 (opencv runtime).
!apt-get -qq install -y ffmpeg libgl1 > /dev/null

In [ ]:
# 3. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Persist model weights to Drive so later sessions skip the downloads.
import os
from pathlib import Path

DRIVE = "/content/drive/MyDrive/occlude"
os.makedirs(f"{DRIVE}/models", exist_ok=True)
os.makedirs(f"{DRIVE}/inputs", exist_ok=True)
os.makedirs(f"{DRIVE}/outputs", exist_ok=True)

# OCCLUDE caches (env-var driven, inherited by the subprocess in cell 7).
os.environ["HF_HOME"]          = f"{DRIVE}/models/hf"
os.environ["INSIGHTFACE_HOME"] = f"{DRIVE}/models/insightface"

# YOLO downloads yolov8n.pt into the cwd; symlink the cached copy.
yolo_cache = f"{DRIVE}/models/yolov8n.pt"
if os.path.exists(yolo_cache) and not os.path.exists("yolov8n.pt"):
    os.symlink(yolo_cache, "yolov8n.pt")

print("caches wired to", DRIVE + "/models")

In [ ]:
# 5. Install occlude[gpu]==1.2.0, repair onnxruntime-gpu, verify CUDA.
#    The uninstall+force-reinstall is REQUIRED: plain `onnxruntime` is an
#    occlude core dep and shadows `onnxruntime-gpu`, so the [gpu] extra
#    alone leaves InsightFace on CPU (~6 fps).
import subprocess, torch

assert torch.cuda.is_available(), "no CUDA - Runtime > Change runtime type > GPU"
print("GPU OK:", torch.cuda.get_device_name(0), flush=True)

subprocess.run("pip install -q -U --index-url https://test.pypi.org/simple/ "
               "--extra-index-url https://pypi.org/simple/ "
               "'occlude[gpu]==1.2.0'", shell=True, check=True)
subprocess.run("pip uninstall -y -q onnxruntime onnxruntime-gpu",
               shell=True, check=False)
subprocess.run("pip install -q --force-reinstall --no-deps onnxruntime-gpu",
               shell=True, check=True)

# Verify in a FRESH process - that is what the occlude subprocess sees.
_verify = r'''
import onnxruntime, torch
from occlude.pipeline.perception import Perception
from occlude.pipeline import video as _v
from occlude.pipeline.io_cuda import cuda_io_available, cuda_io_unavailable_reasons
p = Perception(device="cuda")
seg = next(p.seg_model.parameters())
eps = set()
for m in p.face_app.models.values():
    s = getattr(m, "session", None)
    if s: eps |= set(s.get_providers())
yolo = getattr(p, "_yolo_device", "?")
print("ORT available  :", onnxruntime.get_available_providers())
print("torch device   :", p.device)
print("SegFormer      :", seg.device, seg.dtype)
print("GPU preprocess :", getattr(p, "_gpu_prep", "n/a"))
print("YOLO device    :", yolo, "| fp16:", getattr(p, "_yolo_half", "n/a"))
print("InsightFace EP :", sorted(eps))
print("Blur device    :", _v._BLUR_DEVICE)
print("CUDA video I/O :", cuda_io_available(), cuda_io_unavailable_reasons())
ok = (p.device.type == "cuda"
      and seg.device.type == "cuda"
      and getattr(p, "_gpu_prep", False) is True
      and yolo == 0
      and "CUDAExecutionProvider" in eps
      and _v._BLUR_DEVICE is not None)
print("ALL PARTS ON CUDA:", ok)
assert ok, "a part is NOT on CUDA - inspect the lines above"
'''
chk = subprocess.run(["python", "-c", _verify], capture_output=True, text=True)
print(chk.stdout.strip(), flush=True)
if chk.returncode != 0:
    print(chk.stderr.strip(), flush=True)
    raise SystemExit("GPU verification failed - do not start the run")

In [ ]:
# 6. List what's actually in the Drive inputs/ folder, with sizes, so
#    you can copy the EXACT filename into cell 7 (no guessing, no
#    brittle byte-size assert).
import os
DRIVE = "/content/drive/MyDrive/occlude"
in_dir = f"{DRIVE}/inputs"
assert os.path.isdir(in_dir), f"not found: {in_dir} (is Drive mounted? cell 3)"
for name in sorted(os.listdir(in_dir)):
    p = os.path.join(in_dir, name)
    if os.path.isfile(p):
        print(f"{os.path.getsize(p)/1e6:8.1f} MB  {name}")

In [ ]:
# 7. RUN. OCCLUDE on the original file (music intact). torch.compile is
#    left ON (long run): the one-time ~10-30 frame warm-up amortizes.
import os, shutil, time

DRIVE = "/content/drive/MyDrive/occlude"

# ---- edit these two if the name differs (see cell 6 listing) ----
INPUT_NAME  = "The-Thinking-Game-h264.mp4"
OUTPUT_NAME = "The-Thinking-Game-occluded.mp4"
PERCEPTION_BATCH = 4  # A100/L4 default; try 6 or 8 after a short benchmark if VRAM allows
# -----------------------------------------------------------------

IN  = f"{DRIVE}/inputs/{INPUT_NAME}"
OUT = f"{DRIVE}/outputs/{OUTPUT_NAME}"
assert os.path.exists(IN), (
    f"not found: {IN}\n"
    f"run cell 6 and copy the exact name into INPUT_NAME above."
)
print(f"input: {IN}  ({os.path.getsize(IN)/1e6:.1f} MB)", flush=True)

# Process from local disk, not the Drive FUSE mount: heavy per-frame
# reads/writes over Drive are slow and flaky on long jobs.
shutil.copy(IN, "/content/occ_in.mp4")

print(f">>> occlude (perception batch {PERCEPTION_BATCH}) - first ~10-30 frames "
      "crawl during compile warm-up, then it speeds up", flush=True)
t0 = time.time()
!python -m occlude --input /content/occ_in.mp4 --output /content/occ_out.mp4 --device cuda --perception-batch {PERCEPTION_BATCH}
print(f">>> finished in {(time.time()-t0)/60:.1f} min", flush=True)

assert os.path.exists("/content/occ_out.mp4"), "occlude did not produce an output - see log above"
shutil.copy("/content/occ_out.mp4", OUT)
print("DONE:", os.path.getsize(OUT), "bytes ->", OUT, flush=True)